<div class="blog-language-switch" role="group" aria-label="文章语言">
<a href="/ipynb/Deep-Learning/01-deep-learning-foundations-workflow.html" lang="en" hreflang="en">English</a>
<span aria-current="page">中文</span>
</div>

[返回深度学习总览](Deep-Learning.html)

## **深度学习基础与工作流程** {#deep-learning-foundations-workflow}

深度学习经常被介绍为“使用很多神经网络层的机器学习”。这种描述指出了一个可见的架构特征，却没有触及核心思想。深度学习系统学习的是一种**表示的组合**：每个阶段都把当前样本的描述转换为另一种更有利于最终目标的描述，训练过程再利用数据联合调整这些转换。

本章将建立整套博客后续章节共用的术语和工作流程。在讨论某一种具体架构之前，我们先回答四个问题：

- 深度模型究竟学习了什么？
- 为什么多个非线性转换可能比单个浅层转换更有用？
- 架构、数据、目标函数和训练过程分别引入了哪些假设？
- 在使用更复杂的神经模型替代简单基线之前，需要提供什么证据？

本章的目的并不是论证深度学习永远是最好的工具，而是理解学习表示在什么情况下有价值、需要付出什么成本，以及如何检验这种价值，而不把训练集拟合程度误认为真正的进步。

### **什么是深度学习？** {#what-is-deep-learning}

**深度学习**是机器学习的一个分支，它通过多个参数化转换，联合学习数据表示和预测结果。一个前馈网络可以表示为一系列隐藏状态：

$$
\mathbf{h}^{(0)} = \mathbf{x},
$$

$$
\mathbf{z}^{(\ell)}
= W^{(\ell)}\mathbf{h}^{(\ell-1)}+\mathbf{b}^{(\ell)},
\qquad
\mathbf{h}^{(\ell)}
= \phi^{(\ell)}\!\left(\mathbf{z}^{(\ell)}\right),
\quad \ell=1,\ldots,L,
$$

$$
\widehat{\mathbf{y}} = g\!\left(\mathbf{h}^{(L)}\right).
$$

其中，$\mathbf{x}$ 是输入；$L$ 是可学习转换阶段的数量；$W^{(\ell)}$ 和 $\mathbf{b}^{(\ell)}$ 是可训练的权重与偏置；$\phi^{(\ell)}$ 通常是非线性激活函数或结构化运算；$\mathbf{h}^{(\ell)}$ 是第 $\ell$ 层产生的表示；$g$ 则把最终表示转换为任务所需的输出。所有可训练量合在一起记为参数 $\theta$。

在监督学习中，训练通常最小化如下经验目标：

$$
\widehat{\theta}
= \arg\min_{\theta}
\frac{1}{n}\sum_{i=1}^{n}
\mathcal{L}\!\left(f_{\theta}(\mathbf{x}_i),y_i\right)
+ \lambda\,\Omega(\theta).
$$

$n$ 是训练样本数，$f_{\theta}$ 是完整网络，$\mathcal{L}$ 衡量预测误差，$\Omega$ 是可选的正则项，$\lambda$ 控制其强度。架构决定哪些函数能够被表示；目标函数决定倾向于什么行为；优化器决定如何搜索参数；数据决定模型能够学习哪些差异。这些是不同的设计决策，不能全部模糊地归入“模型”一词。

![深层神经网络让输入依次经过多个隐藏层，然后生成输出。](assets/google-hidden-layers.png){fig-align="center" width="72%" fig-alt="包含输入层、两个隐藏层和输出层的网络结构图。"}

*图片来源：Google for Developers，[Machine Learning Glossary: hidden layer](https://developers.google.com/machine-learning/glossary/fundamentals)，采用 [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) 许可。*

“深”并不存在统一的层数阈值。相对于线性模型，两个隐藏层的网络已经是深层网络；现代基础模型则可能包含数百个计算块。更重要的是，深度应当统计有意义的转换，而不是软件输出中容器的数量。残差块、注意力块、循环状态转换和消息传递步骤都可能以不同方式贡献深度。

深层网络只是在很宽泛的意义上受到生物神经系统启发。它们的计算单元、学习规则、数据需求和连接方式都与大脑存在巨大差异。更有科学意义的定义是计算性的：深度学习通过优化数据或交互产生的目标，学习分层函数与表示。

<details>
<summary><strong>NumPy：最小化的两层前向传播</strong></summary>

~~~python
import numpy as np

# One example with three input features: shape [input_features].
x = np.array([0.4, -1.2, 0.7])

# The first learned transformation maps 3 inputs to 4 hidden features.
W1 = np.array([
    [0.2, -0.1, 0.4],
    [0.7,  0.3, -0.5],
    [-0.6, 0.2, 0.1],
    [0.1,  0.8, 0.2],
])
b1 = np.zeros(4)

# The output transformation maps 4 hidden features to 2 class logits.
W2 = np.array([
    [0.4, -0.3, 0.2, 0.1],
    [-0.2, 0.5, -0.4, 0.7],
])
b2 = np.zeros(2)

# Forward pass: affine transformation -> nonlinearity -> output scores.
hidden_pre_activation = W1 @ x + b1       # shape [4]
hidden = np.maximum(hidden_pre_activation, 0.0)  # ReLU
logits = W2 @ hidden + b2                 # shape [2]

print("hidden representation:", hidden)
print("class logits:", logits)
~~~

</details>

这段代码只执行推理。学习过程还需要损失函数、梯度、参数更新和留出数据评估，这些机制将在第 4 至第 6 章展开。此处最重要的是：输出由一个学习得到的中间表示产生，而不是直接从原始特征得出。

**对比总结。** 神经网络是一类架构；深度学习则是学习分层表示、目标与参数的更宽泛方法。规模很大的网络不等于训练良好的网络，训练良好的网络也不等于能够在目标部署分布上发挥作用的系统。

### **从特征工程到表示学习** {#feature-engineering-to-representation-learning}

所有学习系统都需要表示。图像最初可以表示为像素强度，语音可以表示为波形，文档可以表示为 token 标识符，图则可以表示为节点和边的属性。预测算法无法直接操作真实世界对象，只能操作对象的数值描述。

在经典流水线中，人类设计者通常先指定特征映射 $\psi$，再训练相对简单的预测器：

$$
\mathbf{x}_{\text{raw}}
\xrightarrow{\text{designed }\psi}
\mathbf{r}
\xrightarrow{\text{learned }g_{\omega}}
\widehat{\mathbf{y}}.
$$

例如，图像可以使用边缘直方图，语音可以使用 MFCC 系数，文档可以使用 TF-IDF 向量，表格数据可以使用人工选择的比率。当这些特征能够编码稳定的领域知识、适用于有限数据，并且便于检查错误时，它们可能非常优秀。

表示学习则让特征映射本身可以训练：

$$
\widehat{\mathbf{y}}
= g_{\omega}\!\left(\phi_{\theta}(\mathbf{x})\right).
$$

$\phi_{\theta}$ 学习中间表示，$g_{\omega}$ 利用该表示完成任务。由于 $\theta$ 和 $\omega$ 被联合优化，模型可以保留有助于降低最终损失的差异。在图像分类器中，较早的层可能响应局部对比度与纹理，中间层响应图案或部件，较晚的层则响应与任务有关的配置。在语言任务中，表示可能从 token 身份逐步转向依赖上下文的语法、语义和篇章信息。

<div class="diagram-scroll wide-diagram">

![特征可视化展示了一个训练完成的视觉网络如何从边缘和纹理，逐步形成图案、部件和近似对象级特征。](assets/distill-feature-hierarchy.png){fig-align="center" width="100%" fig-alt="五组分别标记为边缘、纹理、图案、部件和对象的特征可视化。"}

</div>

*图片来源：Olah、Mordvintsev 与 Schubert，[Feature Visualization](https://distill.pub/2017/feature-visualization/)，Distill，2017，采用 [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) 许可。*

这张图需要谨慎解释。特征可视化是经过优化、能够强烈激活某个单元或通道的输入；它提供了模型敏感性的证据，却不能证明一个神经元在字面意义上存储了某个人类概念。表示通常是分布式的，多个特征会协同工作，同一特征也可能对不同模式产生响应。这张图真正有价值的地方是让分层复用变得可见：后续计算建立在前面学到的响应之上。

深度学习并不会消除预处理或人类判断。Tokenisation、归一化、数据增强、采样、标签设计和架构选择仍然会向系统引入结构。“端到端”表示更大一部分映射能够被联合优化，并不表示系统直接接收未经中介的现实，也不表示它能够自行发现正确的目标。

| 设计方式 | 特征来源 | 典型优势 | 典型限制 |
|---|---|---|---|
| 人工特征流水线 | 领域专家和固定转换 | 数据效率高、透明、行为稳定 | 可能丢失信息或需要大量任务特定工作 |
| 学习表示 | 训练目标和数据 | 能让特征适应复杂的高维信号 | 需要充足证据、计算资源和细致的失败分析 |
| 混合流水线 | 领域结构与学习组件 | 将明确约束与灵活表示结合 | 需要验证更多接口与假设 |

**案例。** 医学影像系统可以使用学习得到的视觉特征，同时保留经过校准的采集元数据和临床定义的测量值。欺诈检测系统可以把嵌入表示与明确的交易规则结合。因此，表示学习是一种设计选择，而不是清除所有人工知识的要求。

**对比总结。** 特征工程询问：“预测器应当接收哪些测量？”表示学习询问：“哪些中间测量能够让当前目标更容易完成？”强健的系统通常需要同时回答这两个问题。

### **为什么深度可能有帮助？** {#why-depth-can-help}

深度引入了**组合**。深层模型可以把函数表示为：

$$
f_{\theta}
= f^{(L)}_{\theta_L}
\circ f^{(L-1)}_{\theta_{L-1}}
\circ \cdots
\circ f^{(1)}_{\theta_1}.
$$

当目标过程本身包含可复用阶段时，组合尤其有用。视觉决策可以把边缘组合成轮廓，把轮廓组合成部件，再把部件组合成结构；句子表示可以把 token 组合成上下文关系，再形成任务特定的决策。中间概念不一定与人类标签完全一致，结构优势在于后续计算能够复用早期计算。

非线性至关重要。如果每一层都是线性的，并且没有非线性激活函数，那么：

$$
W^{(3)}W^{(2)}W^{(1)}\mathbf{x}
= W_{\text{equivalent}}\mathbf{x},
$$

因此整个堆叠仍然只是一个线性转换。额外的线性层可能改变优化参数化方式，却无法创建非线性决策边界。激活函数、注意力、门控、归一化交互和其他非线性运算，才让组合能够表示更丰富的函数。

<details>
<summary><strong>NumPy：多个线性层可以合并为一个线性层</strong></summary>

~~~python
import numpy as np

rng = np.random.default_rng(4)
x = rng.normal(size=3)
W1 = rng.normal(size=(5, 3))
W2 = rng.normal(size=(4, 5))
W3 = rng.normal(size=(2, 4))

layered_output = W3 @ (W2 @ (W1 @ x))
equivalent_weight = W3 @ W2 @ W1
single_layer_output = equivalent_weight @ x

assert np.allclose(layered_output, single_layer_output)
print(layered_output)
~~~

</details>

只要宽度足够，浅层网络也可以逼近很广泛的一类函数。通用逼近结果并不表示深度无关紧要。浅层网络可能需要不切实际的宽度，可能无法揭示可复用结构，也可能难以从有限数据中学习。深度分离研究构造了一些函数：紧凑的深层网络能够表示它们，但明显更浅的网络必须使用指数级更多单元才能逼近。[Telgarsky 的深度研究](https://proceedings.mlr.press/v49/telgarsky16.html)就是一个形式化案例。

这类结果证明的是某些函数存在，并不保证增加层数能改善每一个数据集。更深的网络也可能更难优化、增加延迟、放大数据捷径并带来不必要的容量。残差连接、归一化、初始化和现代优化器有助于让深层网络可训练，但它们无法替代任务证据。

| 问题 | 浅层模型 | 更深的模型 |
|---|---|---|
| 能否表示非线性函数？ | 可以，但需要非线性单元和足够宽度 | 可以 |
| 能否复用中间计算？ | 分层能力有限 | 天然支持组合式层级 |
| 优化是否一定更容易？ | 通常更简单 | 缺少架构支持时可能更困难 |
| 准确率是否一定更高？ | 否 | 否 |
| 深度何时更合理？ | 简单关系或低数据场景 | 结构化、高维且具有组合性的信号 |

**对比总结。** 宽度提供并行特征，深度提供顺序复用与组合。真实架构需要平衡两者，并通过验证证据判断增加的结构是否有用。

### **归纳偏置与组合结构** {#inductive-bias-compositional-structure}

**归纳偏置**是在尚未观察所有可能输入输出情况之前，模型对某些解优先于其他解的假设。如果没有这种偏好，就无法实现泛化：许多函数都可以完美拟合有限训练集，却在其他位置给出完全不同的结果。

因为特征可以学习，深度学习有时被描述为没有假设。实际上，每种架构都包含很强的假设：

| 架构 | 重要结构偏置 | 适合的结构 | 可能的不匹配 |
|---|---|---|---|
| 多层感知机 | 向量化后的灵活全局混合 | 固定大小的特征向量 | 除非手动编码，否则忽略空间、时间和图结构 |
| 卷积网络 | 局部连接及与平移有关的权重共享 | 图像、网格和局部信号 | 全局交互需要更深网络或额外机制 |
| 循环网络 | 在时间维度共享状态转换 | 有序序列和流式数据 | 顺序计算限制并行能力与长距离记忆 |
| Transformer | 共享计算块中的内容依赖两两交互 | 上下文序列和多模态 token | 标准注意力处理长上下文时可能很昂贵 |
| 图神经网络 | 感知置换的邻域聚合 | 关系型和几何数据 | 相似局部邻域可能变得不可区分 |
| 状态空间模型 | 可学习状态动力学与高效序列扫描 | 长序列和流式数据 | 将信息压缩到状态可能丢失任务所需交互 |

偏置还来自数据增强、损失函数、优化器、正则化和采样。图像水平翻转假设反射通常不改变标签；因果 mask 假设预测不能查看未来 token；对比学习则假设哪些变换后的样本应共享表示。即使是端到端训练的大模型，也继承了所有这些决策。

因此，实际问题不是模型是否有偏置，而是偏置是否与问题匹配。当局部模式会在不同位置重复时，卷积很有用；如果绝对位置决定含义，而平移本就应改变标签，那么卷积可能不合适。当边表示真实关系时，图模型很有用；随意添加边则可能引入错误的不变性。

**应用场景。** 对于卫星土地覆盖分类，局部空间模式和近似平移不变性使 CNN 成为合理选择。对于交易欺诈，时间顺序、账户关系和不断变化的行为可能需要序列模型、图模型和明确规则的混合系统。只选择“一个深层模型”，却没有识别问题结构，还不能算完成了模型设计。

**对比总结。** 容量描述模型能够表达多少行为；归纳偏置描述哪些行为更容易表达或学习。增加容量无法可靠补偿不匹配的归纳偏置。

### **数据、模型与计算之间的关系** {#data-model-compute-relationship}

现代深度学习由**数据**、**模型容量**与**计算资源**之间的交互塑造。只改善其中一个维度，最终会暴露另一个维度的瓶颈。

**数据**提供模型能够观察到的差异。重要维度包括数量、多样性、标签质量、来源、时效性、重复情况以及关键群体覆盖。一亿条重复或系统性偏差的样本，并不等价于一亿条独立观测。数据转换和采样过程还决定训练时强调哪些证据。

**模型容量**决定哪些映射和表示可以被选择。容量过小的模型会欠拟合；高度灵活的模型则可能记忆噪声、利用捷径或超出部署约束。参数量只是容量指标之一，架构、上下文长度、稀疏性、路由、数值精度和优化过程都会影响实际可用容量。

**计算资源**决定哪些实验和训练轨迹可行。它包括加速器运算、内存、通信、数据加载、能源、工程时间和实际运行时间。更多计算可以支持更大的模型、更多数据、更长的搜索或更稳健的评估，但把资源花在某一项上，也就无法再花在另一项上。

经验缩放定律总结的是特定模型家族、数据集、目标函数和计算范围内观察到的行为，它不是普适的物理定律。Chinchilla 研究表明，在其语言模型设定与固定训练预算下，使模型规模与更多训练数据保持平衡，比单纯扩大模型表现更好。长期有效的结论是资源平衡，而不是把某个常数视为适用于所有模态与部署的永恒规律。参见 [Hoffmann 等人的 Training Compute-Optimal Large Language Models](https://arxiv.org/abs/2203.15556)。

| 瓶颈 | 可观察现象 | 更合理的第一步 |
|---|---|---|
| 数据不足或不匹配 | 切片错误大、验证不稳定、捷径学习 | 改善收集、标签、划分和覆盖 |
| 模型容量不足 | 训练与验证表现都停留在较差水平 | 增加合适容量或更换匹配的架构 |
| 相对证据而言容量过大 | 训练继续改善而验证恶化或波动明显 | 正则化、简化、增强或获取更多证据 |
| 计算不足 | 训练过早停止、实验统计能力不足 | 缩小模型、提高效率或缩小研究问题 |
| 目标函数不合理 | 指标改善但真实决策没有改善 | 重新设计标签、损失、约束或评估 |

最后一行尤其重要，因为数据、模型和计算只会优化给定目标。它们不会判断该目标是否代表用户价值、科学事实、公平性或安全性。问题定义始终是人类与组织的责任。

**对比总结。** 扩展规模可以改善定义良好的学习过程，也会同步放大数据缺陷、目标错配和运行成本。更好的结果来自证据、容量、计算与真实决策之间的平衡。

### **问题形式与应用领域** {#problem-formulations-application-domains}

深度学习并不是单一任务。任务形式需要规定输入单位、输出结构、监督信号、目标函数、评估协议和部署决策。同一种架构可以支持不同任务形式，同一种任务形式也可以由不同架构解决。

| 任务形式 | 输入 -> 输出 | 案例 | 典型输出层或目标 |
|---|---|---|---|
| 分类 | 对象 -> 类别分布 | 医学影像诊断类别 | logits 与交叉熵 |
| 回归 | 对象 -> 连续数值 | 剩余使用寿命 | 标量/向量与回归损失 |
| 密集预测 | 空间输入 -> 空间标签 | 语义分割 | 每个位置的 logits |
| 序列标注 | 序列 -> 每个位置的标签 | 语音帧或 token 标注 | 每步 logits，有时结合结构化解码 |
| 检索/度量学习 | 查询与候选 -> 相似度/排序 | 图文搜索 | 嵌入相似度或对比损失 |
| 自回归生成 | 前缀 -> 下一个元素分布 | 文本、音频或图像 token 生成 | 条件似然 |
| 重建/自监督学习 | 变换后的输入 -> 缺失/原始信息 | 掩码图像建模 | 重建或表示目标 |
| 序列决策 | 观测 -> 动作 | 机器人或游戏 | 价值、策略或 Actor-Critic 目标 |

应用领域包括计算机视觉、语音、语言、推荐、科学建模、医疗、金融、机器人、网络安全和多模态交互。领域名称并不能决定任务形式。医学图像可以被分类、分割、检索、生成，也可以与文本结合；每一种形式需要不同标签、风险控制和证据。

架构与任务也必须分开。Transformer 并不等同于语言模型，它还可以编码图像、预测时间序列、处理生物序列或融合不同模态。CNN 也不等同于分类器，它可以生成密集图或中间特征。明确区分两者，可以避免常见错误：在说明系统应当输出什么之前，就先选择流行架构。

训练前应写出简洁的任务契约：

1. 预测时，一个样本究竟是什么？
2. 当时有哪些信息在法律与运行层面可用？
3. 系统需要什么输出，该输出会如何改变决策？
4. 标签或学习信号如何产生？
5. 哪些错误成本最高？
6. 评估应模拟哪种分布变化？
7. 延迟、内存、隐私和安全约束是什么？

**对比总结。** 模型把数值输入映射为输出；深度学习系统还包括数据过程、学习目标、评估、推理路径和赋予该映射实际意义的决策。

### **什么时候不应选择深度学习？** {#when-deep-learning-not-right-tool}

当输入维度高、有效特征难以手工指定、能够从大量数据学习可复用表示，并且非线性结构足以证明额外复杂度合理时，深度学习最有吸引力。在这些条件之外，简单方法可能更好。

| 场景 | 深度学习可能不是首选的原因 | 候选方案 |
|---|---|---|
| 小规模结构化表格数据 | 神经估计可能不稳定，表示学习优势有限 | 线性模型、树集成、贝叶斯模型 |
| 精确规则定义正确性 | 近似计算会引入没有必要的误差 | 确定性程序、解析器、约束求解器 |
| 标签极少且没有合适预训练 | 模型容量远超可用监督 | 领域特征、概率模型、主动学习 |
| 严格解释或审计要求 | 内部表示可能难以给出充分理由 | 稀疏模型、评分规则、单调模型 |
| 严格延迟、内存或能源预算 | 模型可能无法适配服务环境 | 启发式方法、紧凑经典模型、查找表或缓存 |
| 目标快速变化且标签延迟 | 高成本重训可能跟不上过程变化 | 规则与监控、在线或自适应方法 |
| 因果干预问题 | 仅有预测拟合无法识别干预效果 | 实验设计或因果推断 |

简单模型同时也是诊断工具。如果线性分类器已经达到运行目标，更大的模型就必须证明其维护、推理、监控和失败成本是值得的。如果启发式规则优于神经模型，问题可能出在数据或目标设计上，而不是架构不够复杂。

预训练模型改变了成本计算，因为它把表示学习分摊到了规模更大的源数据集上。即便如此，迁移仍可能因领域变化、继承偏差、许可限制、隐私问题或推理成本而失败。“使用预训练模型”是需要评估的假设，不是免除评估的理由。

**应用场景。** 对于几千行整洁的客户属性数据，梯度提升树是强有力的基线。对于包含复杂视觉变化的数百万商品图像，学习视觉表示更合理。生产系统也可能同时使用两者：神经编码器提供嵌入，较简单且经过校准的模型负责最终受约束决策。

**对比总结。** 正确的问题不是“神经网络能否拟合这个数据集？”它通常可以。真正的问题是它能否创造足够可靠的价值，以证明其证据要求和运行负担合理。

### **可复现的深度学习工作流程** {#reproducible-deep-learning-workflow}

可靠工作流程需要把问题定义、模型选择和最终评估分开。一种实用顺序是：

$$
\text{decision contract}
\rightarrow \text{data audit and split}
\rightarrow \text{baseline}
\rightarrow \text{representation and model}
\rightarrow \text{training}
\rightarrow \text{validation and diagnosis}
\rightarrow \text{locked test evaluation}
\rightarrow \text{deployment and monitoring}.
$$

**决策契约。** 定义预测单位、可用信息、期望行动、错误成本和运行约束，避免优化一个容易测量却无法改善真实结果的代理目标。

**数据审计与划分。** 检查来源、标签、重复、缺失、子群体覆盖和时间结构。在拟合归一化或特征转换之前完成划分。按群体或时间划分通常比随机行划分更接近现实。

**基线。** 从多数类规则、启发式方法、线性模型或小型成熟架构开始。Google 的 [Rules of Machine Learning](https://developers.google.com/machine-learning/guides/rules-of-ml) 强调，简单的第一个模型可以提供基线行为，并在增加架构复杂度之前暴露基础设施问题。

**训练与验证。** 只能利用训练集优化参数。使用验证证据选择架构、超参数、停止点和阈值。在所有选择固定前，测试集必须保持隔离。

**复现记录。** 保存代码版本、环境、数据版本、划分标识、随机种子、配置、日志和最终选定的 checkpoint。PyTorch 明确指出，即使随机种子相同，也不能保证不同版本、平台或 CPU/GPU 执行完全一致。因此，复现声明必须说明环境与确定性设置。参见官方 [PyTorch reproducibility notes](https://docs.pytorch.org/docs/stable/notes/randomness)。

下面的案例在非线性 two-moons 问题上比较线性基线和具有两个隐藏层的 MLP。数据集和网络被刻意保持在较小规模，使每个阶段都容易观察。

<details>
<summary><strong>PyTorch：数据构造、划分与模型定义</strong></summary>

~~~python
from copy import deepcopy
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

SEED = 17


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def make_two_moons(
    n_samples: int = 1_500,
    noise: float = 0.18,
    seed: int = SEED,
) -> tuple[np.ndarray, np.ndarray]:
    """Create a nonlinear two-class dataset without external packages."""
    rng = np.random.default_rng(seed)
    n_outer = n_samples // 2
    n_inner = n_samples - n_outer

    outer_angle = rng.uniform(0.0, np.pi, n_outer)
    inner_angle = rng.uniform(0.0, np.pi, n_inner)
    outer = np.column_stack((np.cos(outer_angle), np.sin(outer_angle)))
    inner = np.column_stack((1.0 - np.cos(inner_angle), 0.5 - np.sin(inner_angle)))

    features = np.vstack((outer, inner))
    labels = np.concatenate((
        np.zeros(n_outer, dtype=np.int64),
        np.ones(n_inner, dtype=np.int64),
    ))
    features += rng.normal(0.0, noise, features.shape)

    order = rng.permutation(n_samples)
    return features[order].astype(np.float32), labels[order]


def split_and_standardize(
    features: np.ndarray,
    labels: np.ndarray,
) -> dict[str, tuple[torch.Tensor, torch.Tensor]]:
    """Split first; fit preprocessing statistics on training data only."""
    train_end = int(0.60 * len(features))
    validation_end = int(0.80 * len(features))
    raw_splits = {
        "train": (features[:train_end], labels[:train_end]),
        "validation": (features[train_end:validation_end], labels[train_end:validation_end]),
        "test": (features[validation_end:], labels[validation_end:]),
    }

    train_features = raw_splits["train"][0]
    mean = train_features.mean(axis=0, keepdims=True)
    std = train_features.std(axis=0, keepdims=True).clip(min=1e-6)

    return {
        name: (
            torch.tensor((x_values - mean) / std, dtype=torch.float32),
            torch.tensor(y_values, dtype=torch.long),
        )
        for name, (x_values, y_values) in raw_splits.items()
    }


class LinearBaseline(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.classifier = nn.Linear(2, 2)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.classifier(inputs)


class SmallMLP(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Linear(16, 2),
        )

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.network(inputs)


seed_everything(SEED)
features, labels = make_two_moons()
splits = split_and_standardize(features, labels)
~~~

</details>

<details>
<summary><strong>PyTorch：训练、checkpoint 选择与评估</strong></summary>

~~~python
@torch.inference_mode()
def accuracy(model, split) -> float:
    features, labels = split
    predictions = model(features).argmax(dim=1)
    return (predictions == labels).float().mean().item()


def train_model(
    model,
    splits,
    learning_rate: float = 0.03,
    maximum_epochs: int = 800,
    patience: int = 80,
):
    """Train on train data and select the checkpoint by validation loss."""
    loss_function = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=1e-4,
    )

    train_features, train_labels = splits["train"]
    validation_features, validation_labels = splits["validation"]
    history = {"train_loss": [], "validation_loss": []}
    best_state = deepcopy(model.state_dict())
    best_validation_loss = float("inf")
    epochs_without_improvement = 0

    for _ in range(maximum_epochs):
        # 1. Update parameters using training examples only.
        model.train()
        optimizer.zero_grad()
        train_logits = model(train_features)
        train_loss = loss_function(train_logits, train_labels)
        train_loss.backward()
        optimizer.step()

        # 2. Measure validation loss without updating parameters.
        model.eval()
        with torch.inference_mode():
            validation_logits = model(validation_features)
            validation_loss = loss_function(validation_logits, validation_labels)

        history["train_loss"].append(train_loss.item())
        history["validation_loss"].append(validation_loss.item())

        # 3. Preserve the checkpoint that generalizes best to validation data.
        if validation_loss.item() < best_validation_loss - 1e-5:
            best_validation_loss = validation_loss.item()
            best_state = deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    model.load_state_dict(best_state)
    return model, history


models = {}
histories = {}
for name, model in {
    "Linear baseline": LinearBaseline(),
    "Two-hidden-layer MLP": SmallMLP(),
}.items():
    seed_everything(SEED)
    trained_model, history = train_model(model, splits)
    models[name] = trained_model
    histories[name] = history

for name, model in models.items():
    print(
        name,
        "validation=", round(accuracy(model, splits["validation"]), 4),
        "test=", round(accuracy(model, splits["test"]), 4),
    )
~~~

</details>

生成本章结果的实际运行得到：

| 模型 | 选定 epoch | 验证准确率 | 测试准确率 |
|---|---:|---:|---:|
| 线性基线 | 98 | 0.873 | 0.887 |
| 两隐藏层 MLP | 54 | 0.977 | 0.963 |

当软件或硬件变化时，具体选中的 epoch 可能不同。因此，工作流程需要保存 checkpoint 与运行环境，而不是把某次结果视为永恒常数。这里的性能差异来自结构：线性分类器只能绘制一条直线边界，非线性 MLP 则可以让边界沿两个月牙弯曲。

![线性基线使用直线决策边界，MLP 则学习贴合两个月牙形状的非线性边界；黄色圆圈表示测试错误。](assets/dl01-linear-vs-mlp.png){fig-align="center" width="100%" fig-alt="线性分类器与两隐藏层 MLP 在 two-moons 数据集上的并排决策边界。"}

*图片由本章可复现的 PyTorch 实验在本地生成。*

这个案例不能证明 MLP 在所有情况下都更优秀。它验证的是一个范围更窄的命题：当决策结构具有非线性并且存在足够证据时，包含非线性中间表示的模型可以消除基线中清晰可见的表示限制。

**对比总结。** 训练脚本并不等于可复现实验。复现还需要固定的问题、不可变的数据划分、记录的配置、基于验证集的决策、锁定测试集的最终评估，以及能够重建运行过程的工件。

### **基线、错误分析与迭代** {#baselines-error-analysis-iteration}

**基线**规定了新方法至少需要改善的行为。常见有效基线包括：

- 用于检查指标解释是否正确的随机预测或多数类预测；
- 表示已有知识的领域启发式规则；
- 用于检验是否真正需要复杂表示学习的线性模型或树模型；
- 用于隔离规模收益的小型神经模型；
- 当前生产系统，包括其延迟和失败行为；
- 删除声称能够带来提升的组件之后的消融模型。

击败很弱的基线只能提供很弱的证据。如果一种新架构只与随机预测比较，实验无法证明其复杂度是必要的。不同方案应尽可能使用相同数据划分、预处理信息、评估协议和调参预算。

聚合指标只是错误分析的起点。应当根据能够表示模型假设或部署风险的条件切分错误，例如类别、群体、来源、时间、输入长度、噪声、置信度、缺失特征或到决策边界的距离。

对于 two-moons 实验，可以在标准化后的第二个特征附近定义重叠区域：

~~~python
@torch.inference_mode()
def slice_report(model, split):
    features, labels = split
    predictions = model(features).argmax(dim=1)

    # This band contains points near the region where a straight boundary fails.
    overlap_mask = features[:, 1].abs() < 0.45

    return {
        "overall": (predictions == labels).float().mean().item(),
        "overlap_band": (
            (predictions[overlap_mask] == labels[overlap_mask]).float().mean().item()
        ),
        "outside_band": (
            (predictions[~overlap_mask] == labels[~overlap_mask]).float().mean().item()
        ),
    }


for name, model in models.items():
    print(name, slice_report(model, splits["test"]))
~~~

| 模型 | 整体 | 重叠区域 | 区域外部 |
|---|---:|---:|---:|
| 线性基线 | 0.887 | 0.705 | 0.951 |
| 两隐藏层 MLP | 0.963 | 1.000 | 0.951 |

切片结果比整体提升揭示了更多信息。在选定区域之外，两个模型的表现近似；MLP 的提升主要集中在线性假设错误的区域。这个结果支持我们提出的机制解释。如果提升只发生在某个无关且简单的区域，就需要修改解释。

一种实用的错误分类方式是：

| 错误来源 | 诊断证据 | 典型下一步 |
|---|---|---|
| 数据或标签错误 | 相互矛盾的重复样本、标注不确定 | 修复数据或保留不确定性 |
| 覆盖错误 | 某个来源或群体表现较差 | 收集代表性样本或限制使用范围 |
| 表示/架构错误 | 持续无法处理某种结构模式 | 改变归纳偏置或表示 |
| 优化错误 | 训练损失较高、梯度不稳定 | 检查尺度、初始化、优化器和数值计算 |
| 泛化错误 | 训练损失低但验证表现弱 | 正则化、简化、增强或增加证据 |
| 目标错误 | 指标上升而实际行为变差 | 重新设计损失、标签、约束或决策规则 |
| 评估错误 | 数据泄漏、反复测试调参、切片错误 | 在修改模型前重建评估协议 |
| 部署错误 | 离线与在线不一致、延迟或漂移 | 验证完整服务路径并进行监控 |

迭代过程应当遵循证据：

$$
\text{observe failure}
\rightarrow \text{form a mechanism-level hypothesis}
\rightarrow \text{change one relevant factor}
\rightarrow \text{run a controlled comparison}
\rightarrow \text{update the error map}.
$$

同时修改架构、数据增强、优化器、数据和指标，可能提高分数，却无法说明原因。受控实验让模型开发积累为可复用知识，而不是一连串昂贵猜测。

**对比总结。** 优化询问如何降低训练目标；错误分析询问剩余失败究竟来自数据、表示、优化、目标、评估还是部署。第二个问题决定下一步应当修改什么。

### **本章对比与总结** {#chapter-comparison-summary}

理解深度学习最合适的方式，是把它视为一种在明确架构、数据、目标和计算假设下学习组合表示的方法。它的优势并不是让人类停止设计系统，而是让许多有用的中间特征能够根据证据被联合调整，并在复杂转换中复用。

| 维度 | 经典特征方法 | 深度表示学习方法 |
|---|---|---|
| 中间特征 | 主要在拟合前指定 | 主要结合目标学习 |
| 典型模型 | 线性、核、树、概率模型 | 分层可微架构 |
| 数据需求 | 通常适合较小的结构化数据集 | 通常受益于大规模数据或预训练数据 |
| 计算与工程 | 通常较低 | 可能高得多 |
| 结构假设 | 特征设计和模型家族 | 架构、目标、数据与训练过程 |
| 调试 | 特征和系数可能更容易检查 | 需要表示、切片和失败分析 |
| 最适合 | 特征清晰、数据有限、约束严格 | 高维信号与可复用复杂结构 |

本章的主要结论是：

1. 深层网络是可学习转换的组合，而不仅仅是大量参数的集合。
2. 表示学习把一部分特征设计转移到优化过程中，但不会消除预处理、任务定义和人类假设。
3. 深度可以高效表示组合函数，但增加层数不会自动产生收益；没有非线性运算的多层线性网络仍等价于一个线性映射。
4. 架构是一种归纳偏置。CNN、循环模型、Transformer、图网络和状态空间模型偏好不同结构。
5. 数据、模型容量、计算与目标必须保持平衡，扩大其中一项无法修复其他项的缺陷。
6. 深度学习必须与可信的简单基线竞争，并证明其运行成本合理。
7. 可信工作流程应当分离训练、验证和测试决策，记录复现信息，并围绕错误假设迭代，而不是追逐架构潮流。

下一章将介绍用于精确表达这些分层转换的 Tensor 运算、计算图和 PyTorch 抽象。
